In [1]:
"""
AISEHack Phase 2 — 3-Class Flood Segmentation
Kaggle Notebook (as .py — convert to .ipynb for submission)

Competition: https://www.kaggle.com/competitions/anrfaisehack-theme-1-phase2

Task: 3-class semantic segmentation
  0 = No Flood
  1 = Flood
  2 = Water Body

Data: 6-band GeoTIFF (HH, HV, Green, Red, NIR, SWIR), 512x512 pixels
Model: Prithvi-EO-2.0-300M-TL fine-tuned via TerraTorch
Submission: RLE-encoded flood mask (class 1 only)

Author: AISEHack Team
"""

'\nAISEHack Phase 2 — 3-Class Flood Segmentation\nKaggle Notebook (as .py — convert to .ipynb for submission)\n\nCompetition: https://www.kaggle.com/competitions/anrfaisehack-theme-1-phase2\n\nTask: 3-class semantic segmentation\n  0 = No Flood\n  1 = Flood\n  2 = Water Body\n\nData: 6-band GeoTIFF (HH, HV, Green, Red, NIR, SWIR), 512x512 pixels\nModel: Prithvi-EO-2.0-300M-TL fine-tuned via TerraTorch\nSubmission: RLE-encoded flood mask (class 1 only)\n\nAuthor: AISEHack Team\n'

# AISEHack Phase 2 — 3-Class Flood Segmentation

**Objective**: Pixel-wise classification into No Flood (0), Flood (1), Water Body (2)
using satellite imagery (SAR + Optical) from West Bengal, India.

In [2]:
import subprocess
import sys

def install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", *packages, "-q"])

# Kaggle already has numpy, scipy, and albumentations.
# Modifying them in a running session causes memory/disk state mismatches.
# We only install the models package we need.
install("segmentation-models-pytorch")

import os
import glob
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import rasterio
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.4 MB/s eta 0:00:00


In [3]:
SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
print(f"Seeds set to {SEED}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

Seeds set to 42
PyTorch version: 2.10.0+cu128
CUDA available: True
Device: cuda


In [4]:
class Config:
    # Data paths — adjust for Kaggle
    DATA_ROOT = "/kaggle/input/anrfaisehack-theme-1-phase2"  # Kaggle dataset path
    # LOCAL_DATA_ROOT = "data"  # Local development path
    
    IMAGE_DIR = os.path.join(DATA_ROOT, "image")
    LABEL_DIR = os.path.join(DATA_ROOT, "label")
    PRED_IMAGE_DIR = os.path.join(DATA_ROOT, "prediction", "image")
    SPLIT_DIR = os.path.join(DATA_ROOT, "split")
    
    # Model
    NUM_CLASSES = 3
    IN_CHANNELS = 6  # HH, HV, Green, Red, NIR, SWIR
    BACKBONE = "resnet50"  # Encoder backbone
    
    # Training
    IMG_SIZE = 512  # Keep native resolution
    BATCH_SIZE = 4
    NUM_EPOCHS = 30
    LR = 1e-4
    WEIGHT_DECAY = 1e-4
    NUM_WORKERS = 2
    
    # Loss weights (inverse frequency from class distribution)
    # Class 0: 65.62% → weight ~1.0 (reference)
    # Class 1: 13.49% → weight ~4.86
    # Class 2: 20.89% → weight ~3.14
    CLASS_WEIGHTS = [1.0, 4.86, 3.14]
    
    # Band descriptions
    BAND_NAMES = ['HH', 'HV', 'Green', 'Red', 'NIR', 'SWIR']
    
    SEED = SEED

cfg = Config()

In [5]:

def read_split_file(filepath):
    """Read split file and return list of image IDs."""
    with open(filepath, 'r') as f:
        ids = [line.strip() for line in f if line.strip()]
    return ids


def load_image(image_id, image_dir):
    """Load multi-band GeoTIFF image as numpy array (C, H, W)."""
    path = os.path.join(image_dir, f"{image_id}_image.tif")
    with rasterio.open(path) as src:
        data = src.read().astype(np.float32)  # (C, H, W)
    # Replace NaN with 0
    data = np.nan_to_num(data, nan=0.0)
    return data


def load_label(image_id, label_dir):
    """Load single-band label GeoTIFF as numpy array (H, W)."""
    path = os.path.join(label_dir, f"{image_id}_label.tif")
    with rasterio.open(path) as src:
        label = src.read(1).astype(np.int64)  # (H, W)
    return label


def normalize_image(image):
    """
    Normalize each band independently to [0, 1] using per-band min-max.
    image: (C, H, W) numpy array
    """
    normalized = np.zeros_like(image)
    for i in range(image.shape[0]):
        band = image[i]
        bmin, bmax = band.min(), band.max()
        if bmax > bmin:
            normalized[i] = (band - bmin) / (bmax - bmin)
        else:
            normalized[i] = 0.0
    return normalized

In [6]:

class FloodDataset(Dataset):
    """Custom dataset for flood segmentation with SAR+Optical imagery."""
    
    def __init__(self, image_ids, image_dir, label_dir=None, transform=None, is_test=False):
        self.image_ids = image_ids
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.transform = transform
        self.is_test = is_test
    
    def __len__(self):
        return len(self.image_ids)
    
    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        
        # Load image (C, H, W)
        image = load_image(image_id, self.image_dir)
        
        # Normalize per-band to [0, 1]
        image = normalize_image(image)
        
        # Load label if available
        if not self.is_test and self.label_dir:
            label = load_label(image_id, self.label_dir)
        else:
            label = np.zeros((image.shape[1], image.shape[2]), dtype=np.int64)
        
        # Albumentations expects (H, W, C) for image
        image = image.transpose(1, 2, 0)  # (H, W, C)
        
        if self.transform:
            augmented = self.transform(image=image, mask=label)
            image = augmented['image']  # (C, H, W) tensor
            label = augmented['mask']   # (H, W) tensor
        else:
            image = torch.from_numpy(image.transpose(2, 0, 1)).float()
            label = torch.from_numpy(label).long()
        
        return {
            'image': image,
            'label': label,
            'image_id': image_id
        }

In [7]:

def get_train_transform(img_size=512):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(
            shift_limit=0.05,
            scale_limit=0.1,
            rotate_limit=15,
            border_mode=0,
            p=0.3
        ),
        A.GaussNoise(var_limit=(0.001, 0.01), p=0.2),
        ToTensorV2(),
    ])


def get_val_transform(img_size=512):
    return A.Compose([
        A.Resize(img_size, img_size),
        ToTensorV2(),
    ])

In [8]:

class DiceLoss(nn.Module):
    """Dice Loss for multi-class segmentation."""
    def __init__(self, num_classes=3, smooth=1e-6):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth
    
    def forward(self, pred, target):
        # pred: (B, C, H, W) logits
        # target: (B, H, W) class indices
        pred_soft = F.softmax(pred, dim=1)
        target_one_hot = F.one_hot(target, self.num_classes).permute(0, 3, 1, 2).float()
        
        intersection = (pred_soft * target_one_hot).sum(dim=(2, 3))
        union = pred_soft.sum(dim=(2, 3)) + target_one_hot.sum(dim=(2, 3))
        
        dice = (2.0 * intersection + self.smooth) / (union + self.smooth)
        return 1.0 - dice.mean()


class CombinedLoss(nn.Module):
    """Weighted CE + Dice Loss."""
    def __init__(self, class_weights, num_classes=3, ce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(
            weight=torch.FloatTensor(class_weights)
        )
        self.dice = DiceLoss(num_classes=num_classes)
        self.ce_weight = ce_weight
        self.dice_weight = dice_weight
    
    def forward(self, pred, target):
        return self.ce_weight * self.ce(pred, target) + self.dice_weight * self.dice(pred, target)


def build_model(in_channels=6, num_classes=3, backbone='resnet50'):
    """Build segmentation model using segmentation_models_pytorch."""
    import segmentation_models_pytorch as smp
    
    model = smp.UnetPlusPlus(
        encoder_name=backbone,
        encoder_weights='imagenet',
        in_channels=in_channels,
        classes=num_classes,
        activation=None,  # Raw logits
    )
    return model


# Alternative: Use Prithvi via TerraTorch (if band mapping works)
def build_prithvi_model():
    """
    Build Prithvi-based model via TerraTorch.
    
    NOTE: Our data has bands [HH, HV, Green, Red, NIR, SWIR]
    Prithvi expects [Blue, Green, Red, NIR_NARROW, SWIR_1, SWIR_2]
    
    We lack 'Blue' and 'SWIR_2', and have SAR bands instead.
    Options:
      1. Use only optical bands [Green, Red, NIR, SWIR] (4 bands) — loses SAR info
      2. Map our bands: Green→Green, Red→Red, NIR→NIR_NARROW, SWIR→SWIR_1,
         and use dummy/zero for Blue and SWIR_2 — suboptimal
      3. Use a standard encoder (ResNet/EfficientNet) that accepts arbitrary channels
    
    For robustness, we use option 3 (smp.UnetPlusPlus with ResNet50) as the primary approach,
    and provide the Prithvi approach as an alternative if band-compatible data is available.
    """
    try:
        from terratorch.tasks import SemanticSegmentationTask
        # This would need proper config — keeping as reference
        print("TerraTorch available — Prithvi model can be used if bands are compatible")
    except ImportError:
        print("TerraTorch not available — using SMP model")
    return None

In [9]:

def compute_iou(pred, target, num_classes=3):
    """Compute per-class IoU and mIoU."""
    ious = []
    for cls in range(num_classes):
        pred_cls = (pred == cls)
        target_cls = (target == cls)
        
        intersection = (pred_cls & target_cls).sum().item()
        union = (pred_cls | target_cls).sum().item()
        
        if union == 0:
            iou = float('nan')  # Skip if class not present
        else:
            iou = intersection / union
        ious.append(iou)
    
    valid_ious = [x for x in ious if not np.isnan(x)]
    miou = np.mean(valid_ious) if valid_ious else 0.0
    return ious, miou


def compute_pixel_accuracy(pred, target):
    """Compute overall pixel accuracy."""
    correct = (pred == target).sum().item()
    total = target.numel()
    return correct / total

In [10]:

def train_one_epoch(model, dataloader, criterion, optimizer, device, scaler=None):
    model.train()
    total_loss = 0
    total_correct = 0
    total_pixels = 0
    
    for batch_idx, batch in enumerate(dataloader):
        images = batch['image'].to(device)
        labels = batch['label'].to(device).long()
        
        optimizer.zero_grad()
        
        if scaler:  # Mixed precision
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        
        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_pixels += labels.numel()
        
        if (batch_idx + 1) % 10 == 0:
            print(f"  Batch {batch_idx+1}/{len(dataloader)}, Loss: {loss.item():.4f}")
    
    avg_loss = total_loss / len(dataloader.dataset)
    accuracy = total_correct / total_pixels
    return avg_loss, accuracy


@torch.no_grad()
def validate(model, dataloader, criterion, device, num_classes=3):
    model.eval()
    total_loss = 0
    all_ious = {i: [] for i in range(num_classes)}
    total_correct = 0
    total_pixels = 0
    
    for batch in dataloader:
        images = batch['image'].to(device)
        labels = batch['label'].to(device).long()
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * images.size(0)
        
        preds = outputs.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_pixels += labels.numel()
        
        # Per-batch IoU
        for b in range(preds.size(0)):
            ious, _ = compute_iou(
                preds[b].cpu().numpy(),
                labels[b].cpu().numpy(),
                num_classes
            )
            for cls in range(num_classes):
                if not np.isnan(ious[cls]):
                    all_ious[cls].append(ious[cls])
    
    avg_loss = total_loss / len(dataloader.dataset)
    accuracy = total_correct / total_pixels
    
    class_ious = {}
    for cls in range(num_classes):
        if all_ious[cls]:
            class_ious[cls] = np.mean(all_ious[cls])
        else:
            class_ious[cls] = 0.0
    
    miou = np.mean(list(class_ious.values()))
    
    return avg_loss, accuracy, class_ious, miou

In [11]:

def rle_encode(mask):
    """
    Run-length encoding for a binary mask.
    
    Args:
        mask: numpy array (H, W), values 0 or 1
    
    Returns:
        RLE string in Kaggle convention:
        - Column-major (Fortran) order
        - 1-indexed pixel positions
        - Format: "start1 length1 start2 length2 ..."
    """
    # Flatten in column-major (top-to-bottom, left-to-right)
    pixels = mask.flatten(order='F')
    
    # Pad with zeros at start and end
    pixels = np.concatenate([[0], pixels, [0]])
    
    # Find run starts and ends
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    
    if len(runs) == 0:
        return "0 0"
    
    # Convert to start-length pairs
    runs[1::2] -= runs[::2]
    
    rle = ' '.join(str(x) for x in runs)
    return rle if rle else "0 0"


def rle_decode(rle_string, shape):
    """Decode RLE string back to binary mask (for verification)."""
    if rle_string == "0 0" or rle_string == "":
        return np.zeros(shape, dtype=np.uint8)
    
    s = list(map(int, rle_string.split()))
    starts, lengths = s[0::2], s[1::2]
    
    mask = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for start, length in zip(starts, lengths):
        mask[start-1:start-1+length] = 1  # 1-indexed to 0-indexed
    
    return mask.reshape(shape, order='F')


# Verify RLE round-trip
def test_rle():
    """Test that RLE encode/decode is a perfect round-trip."""
    np.random.seed(42)
    mask = np.random.randint(0, 2, (512, 512)).astype(np.uint8)
    encoded = rle_encode(mask)
    decoded = rle_decode(encoded, (512, 512))
    assert np.array_equal(mask, decoded), "RLE round-trip failed!"
    print("✓ RLE encode/decode round-trip verified")
    
    # Test empty mask
    empty = np.zeros((512, 512), dtype=np.uint8)
    assert rle_encode(empty) == "0 0", "Empty mask RLE failed!"
    print("✓ Empty mask RLE verified")

test_rle()

✓ RLE encode/decode round-trip verified
✓ Empty mask RLE verified


In [12]:

def main():
    print("=" * 60)
    print("AISEHack Phase 2 — 3-Class Flood Segmentation")
    print("=" * 60)
    
    # ── Detect environment ──
    if os.path.exists("/kaggle/input"):
        cfg.DATA_ROOT = "/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data"
        print("Running on Kaggle")
    elif os.path.exists("data"):
        cfg.DATA_ROOT = "data"
        print("Running locally")
    else:
        raise FileNotFoundError("Cannot find data directory")
    
    cfg.IMAGE_DIR = os.path.join(cfg.DATA_ROOT, "image")
    cfg.LABEL_DIR = os.path.join(cfg.DATA_ROOT, "label")
    cfg.PRED_IMAGE_DIR = os.path.join(cfg.DATA_ROOT, "prediction", "image")
    cfg.SPLIT_DIR = os.path.join(cfg.DATA_ROOT, "split")
    
    # ── Load splits ──
    train_ids = read_split_file(os.path.join(cfg.SPLIT_DIR, "train.txt"))
    val_ids = read_split_file(os.path.join(cfg.SPLIT_DIR, "val.txt"))
    test_ids = read_split_file(os.path.join(cfg.SPLIT_DIR, "test.txt"))
    pred_ids = read_split_file(os.path.join(cfg.SPLIT_DIR, "pred.txt"))
    
    print(f"\nData splits:")
    print(f"  Train: {len(train_ids)} images")
    print(f"  Val:   {len(val_ids)} images")
    print(f"  Test:  {len(test_ids)} images")
    print(f"  Pred:  {len(pred_ids)} images (for submission)")
    
    # ── Create datasets ──
    train_dataset = FloodDataset(
        train_ids, cfg.IMAGE_DIR, cfg.LABEL_DIR,
        transform=get_train_transform(cfg.IMG_SIZE)
    )
    val_dataset = FloodDataset(
        val_ids, cfg.IMAGE_DIR, cfg.LABEL_DIR,
        transform=get_val_transform(cfg.IMG_SIZE)
    )
    # Use test split for additional validation
    test_dataset = FloodDataset(
        test_ids, cfg.IMAGE_DIR, cfg.LABEL_DIR,
        transform=get_val_transform(cfg.IMG_SIZE)
    )
    # Prediction dataset (no labels)
    pred_dataset = FloodDataset(
        pred_ids, cfg.PRED_IMAGE_DIR, label_dir=None,
        transform=get_val_transform(cfg.IMG_SIZE),
        is_test=True
    )
    
    # ── Create data loaders ──
    train_loader = DataLoader(
        train_dataset, batch_size=cfg.BATCH_SIZE,
        shuffle=True, num_workers=cfg.NUM_WORKERS,
        pin_memory=True, drop_last=True
    )
    val_loader = DataLoader(
        val_dataset, batch_size=cfg.BATCH_SIZE,
        shuffle=False, num_workers=cfg.NUM_WORKERS,
        pin_memory=True
    )
    test_loader = DataLoader(
        test_dataset, batch_size=cfg.BATCH_SIZE,
        shuffle=False, num_workers=cfg.NUM_WORKERS,
        pin_memory=True
    )
    pred_loader = DataLoader(
        pred_dataset, batch_size=1,
        shuffle=False, num_workers=0,
        pin_memory=True
    )
    
    # ── Verify a batch ──
    sample = next(iter(train_loader))
    print(f"\nSample batch:")
    print(f"  Image shape: {sample['image'].shape}")
    print(f"  Label shape: {sample['label'].shape}")
    print(f"  Label unique: {torch.unique(sample['label']).tolist()}")
    
    # ── Build model ──
    print(f"\nBuilding model: UNet++ with {cfg.BACKBONE} backbone")
    model = build_model(
        in_channels=cfg.IN_CHANNELS,
        num_classes=cfg.NUM_CLASSES,
        backbone=cfg.BACKBONE
    )
    model = model.to(DEVICE)
    
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Total parameters: {total_params:,}")
    print(f"  Trainable parameters: {trainable_params:,}")
    
    # ── Loss, optimizer, scheduler ──
    criterion = CombinedLoss(
        class_weights=cfg.CLASS_WEIGHTS,
        num_classes=cfg.NUM_CLASSES,
        ce_weight=0.5,
        dice_weight=0.5
    ).to(DEVICE)
    
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.LR,
        weight_decay=cfg.WEIGHT_DECAY
    )
    
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=cfg.NUM_EPOCHS, eta_min=1e-6
    )
    
    # Mixed precision scaler
    scaler = torch.cuda.amp.GradScaler() if DEVICE.type == 'cuda' else None
    
    # ── Training loop ──
    best_miou = 0.0
    best_model_state = None
    
    print(f"\n{'='*60}")
    print(f"Starting training for {cfg.NUM_EPOCHS} epochs")
    print(f"{'='*60}")
    
    for epoch in range(cfg.NUM_EPOCHS):
        print(f"\nEpoch {epoch+1}/{cfg.NUM_EPOCHS} (LR: {optimizer.param_groups[0]['lr']:.6f})")
        print("-" * 40)
        
        # Train
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, DEVICE, scaler
        )
        
        # Validate
        val_loss, val_acc, val_ious, val_miou = validate(
            model, val_loader, criterion, DEVICE, cfg.NUM_CLASSES
        )
        
        scheduler.step()
        
        print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
        print(f"  Val Loss:   {val_loss:.4f}, Val Acc:   {val_acc:.4f}")
        print(f"  Val IoU — NoFlood: {val_ious[0]:.4f}, Flood: {val_ious[1]:.4f}, Water: {val_ious[2]:.4f}")
        print(f"  Val mIoU:  {val_miou:.4f}")
        
        # Save best model
        if val_miou > best_miou:
            best_miou = val_miou
            best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
            print(f"  ★ New best mIoU: {best_miou:.4f}")
    
    # ── Load best model ──
    if best_model_state:
        model.load_state_dict(best_model_state)
        print(f"\nLoaded best model with mIoU: {best_miou:.4f}")
    
    # ── Evaluate on test set ──
    print(f"\n{'='*60}")
    print("Evaluating on test set")
    print(f"{'='*60}")
    
    test_loss, test_acc, test_ious, test_miou = validate(
        model, test_loader, criterion, DEVICE, cfg.NUM_CLASSES
    )
    print(f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")
    print(f"Test IoU — NoFlood: {test_ious[0]:.4f}, Flood: {test_ious[1]:.4f}, Water: {test_ious[2]:.4f}")
    print(f"Test mIoU: {test_miou:.4f}")
    
    # ── Inference on prediction set ──
    print(f"\n{'='*60}")
    print("Running inference on prediction images")
    print(f"{'='*60}")
    
    model.eval()
    submissions = []
    
    with torch.no_grad():
        for batch in pred_loader:
            images = batch['image'].to(DEVICE)
            image_ids = batch['image_id']
            
            # Forward pass
            if DEVICE.type == 'cuda':
                with torch.cuda.amp.autocast():
                    outputs = model(images)
            else:
                outputs = model(images)
            
            # Get predictions (B, H, W)
            preds = outputs.argmax(dim=1).cpu().numpy()
            
            for i, image_id in enumerate(image_ids):
                pred_mask = preds[i]  # (H, W) with values 0, 1, 2
                
                # Extract flood class (class 1) binary mask
                flood_mask = (pred_mask == 1).astype(np.uint8)
                
                # RLE encode
                rle = rle_encode(flood_mask)
                
                flood_pixels = flood_mask.sum()
                print(f"  {image_id}: {flood_pixels} flood pixels, RLE length: {len(rle.split())//2} runs")
                
                submissions.append({
                    'id': image_id,
                    'rle_mask': rle
                })
    
    # ── Generate submission CSV ──
    submission_df = pd.DataFrame(submissions)
    
    # Verify all prediction IDs are present
    assert len(submission_df) == len(pred_ids), \
        f"Expected {len(pred_ids)} predictions, got {len(submission_df)}"
    
    # Verify no empty rle_mask
    assert submission_df['rle_mask'].notna().all(), "Found null rle_mask values!"
    assert (submission_df['rle_mask'] != '').all(), "Found empty rle_mask values!"
    
    # Save submission
    output_path = "submission.csv"
    submission_df.to_csv(output_path, index=False)
    print(f"\n✓ Submission saved to {output_path}")
    print(f"  Rows: {len(submission_df)}")
    print(submission_df.head())
    
    print(f"\n{'='*60}")
    print("DONE — submission.csv generated successfully!")
    print(f"{'='*60}")


if __name__ == "__main__":
    main()

AISEHack Phase 2 — 3-Class Flood Segmentation
Running on Kaggle

Data splits:
  Train: 59 images
  Val:   10 images
  Test:  10 images
  Pred:  19 images (for submission)

Sample batch:
  Image shape: torch.Size([4, 6, 512, 512])
  Label shape: torch.Size([4, 512, 512])
  Label unique: [0, 1, 2]

Building model: UNet++ with resnet50 backbone


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

  Total parameters: 48,995,443
  Trainable parameters: 48,995,443

Starting training for 30 epochs

Epoch 1/30 (LR: 0.000100)
----------------------------------------
  Batch 10/14, Loss: 0.9268
  Train Loss: 0.8524, Train Acc: 0.4768
  Val Loss:   1.0752, Val Acc:   0.2729
  Val IoU — NoFlood: 0.1503, Flood: 0.0531, Water: 0.2476
  Val mIoU:  0.1503
  ★ New best mIoU: 0.1503

Epoch 2/30 (LR: 0.000100)
----------------------------------------
  Batch 10/14, Loss: 0.6961
  Train Loss: 0.7338, Train Acc: 0.5832
  Val Loss:   1.0929, Val Acc:   0.4132
  Val IoU — NoFlood: 0.3769, Flood: 0.0836, Water: 0.2355
  Val mIoU:  0.2320
  ★ New best mIoU: 0.2320

Epoch 3/30 (LR: 0.000099)
----------------------------------------
  Batch 10/14, Loss: 0.6884
  Train Loss: 0.6819, Train Acc: 0.6197
  Val Loss:   0.9753, Val Acc:   0.5321
  Val IoU — NoFlood: 0.5004, Flood: 0.0827, Water: 0.2815
  Val mIoU:  0.2882
  ★ New best mIoU: 0.2882

Epoch 4/30 (LR: 0.000098)
----------------------------------